# HW3: Campus Edge

**COMPSS 211A | Fall 2026 | Student copy**

In this homework you work with real 311 service requests from the streets around campus. You will send a request to the city's API, check that two saved files hold the same data, and count the requests with SQL. Then you will join two downloads of the data and find out why one has fewer cases.

**Due:** Sunday, October 25, 2026 at 11:59 p.m.

Fill in every cell marked "Your turn" and answer the written questions. Keep the variable names we ask for, because we look for them when grading.

## Scenario

311 is Berkeley's non-emergency service line. People call it to report things like illegal dumping, noise, or a tree that needs pruning. Each report becomes a *case* with its own ID number.

You will find out what kinds of requests people make near campus. The data for Tasks 2 to 4 is saved in `data/`. Task 1 sends one small request to the city's API, which needs no key.

## What you will practice

- Sending an API request, checking its status, and turning the response into a table (Week 6).
- Checking that two files hold the same data.
- Writing SQL queries that filter, count, and sort (Week 5).
- Joining two tables on a shared column in SQL (Week 5).
- Saying what a dataset can and cannot tell you.

## Helpful references

- Lab 5 (`lab/lab05_sql_from_python.ipynb`) has every kind of query you need for Tasks 3 and 4: `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT`, `IS NULL`, and `LEFT JOIN`. The `sql("""...""")` helper in this notebook works the same way.
- The SQL drills in the [practice studio](https://macss-berkeley.github.io/compss-211a/practice/) check your answer as you go.
- Lab 6 (`lab/lab06_openalex_api_acquisition.ipynb`) and the [Week 6 API workspace](https://macss-berkeley.github.io/compss-211a/interactives/week06-api-workspace.html) cover requests, status codes, and paging.
- Load `case_id` as text, with `dtype={"case_id": str}`. An ID is a label, like a phone number, not a number you do math with.

## AI and collaboration policy

You may use AI to help you understand an error message, as long as you say so in a note in your notebook. Write the code and the written answers yourself.

In [ ]:
from pathlib import Path
import json
import sqlite3
import sys

import pandas as pd
import requests
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Send an API request

The data comes from the City of Berkeley's API. A request is a web address plus *parameters*, which tell the server which records you want. In Python the parameters go in a dictionary. `$where`, `$order`, and `$limit` work like `WHERE`, `ORDER BY`, and `LIMIT` in SQL.

This API sends at most 1,000 records per request. To get more, you send the request again with `$offset`, which says how many records to skip. This is called *paging*. The saved files for Tasks 2 to 4 have 1,896 records, so they took two requests: one with `$offset` 0 and one with `$offset` 1000.

In [ ]:
API_ENDPOINT = "https://data.cityofberkeley.info/resource/p88g-6gs2.json"

query_params = {
    "$where": "date_opened >= '2025-01-01' AND date_opened < '2026-01-01'",  # cases opened in 2025
    "$order": "case_id",  # sort by case ID
    "$limit": 5,          # only 5 records, to keep the request small
}

query_params

In [ ]:
# Your turn: send the request.
# 1. Send it with requests.get(API_ENDPOINT, params=query_params, timeout=30). Save the result as response.
# 2. Print response.status_code. 200 means the request worked.
# 3. Save response.json() as records. Print how many records came back, and show the first one.


If the request fails (no connection, or a status other than 200), run the line below instead and say so in your response. It reads the first five records from the saved file.

```python
records = json.loads(course_data_path("hw3_campus_edge_311_2025.json").read_text())[:5]
```

In [ ]:
# Your turn: make a table from the records.
# 1. Loop over records. For each one, add a dictionary with case_id, date_opened, and case_request to rows.
# 2. Make a DataFrame called live_table from rows and show it.
rows = []


### Your response


Answer in two or three sentences:

1. Which records does `$where` keep, and which does it leave out?
2. If `$limit` were 500 instead of 1000, how many requests would it take to get all 1,896 records? What `$offset` would each one use?


> Write your response here, then delete this line.

## 2. Load the saved files

The folder `data/` has the August 23 download saved in two formats. CSV is what you get when you export a table. JSON is what the API sends back. The cell below loads both.

The two files should hold the same records. Check that they do before you use them.

In [ ]:
# Supplied: load the same records from both files.
csv_rows = pd.read_csv(course_data_path("hw3_campus_edge_311_2025.csv"), dtype={"case_id": str})
json_rows = pd.read_json(course_data_path("hw3_campus_edge_311_2025.json"), dtype={"case_id": str})
print("csv", csv_rows.shape)
print("json", json_rows.shape)

In [ ]:
# Your turn: check that the two files match.
# 1. Make same_ids: True if both tables have the same set of case_id values.
# 2. Make csv_missing and json_missing: how many date_closed values are missing in each table.
#    Use .isna().sum(). Print all three.


# Your turn: the two missing counts are different. A case that is still open has no closing date.
# 3. Show the first few date_closed values of each table. How does each format write "no date"?
# 4. Fix json_rows so pandas counts its blank dates as missing. .replace("", pd.NA) turns empty
#    text into a missing value. Print the missing count again. It should match the CSV.


## 3. Count the request types with SQL

Which kinds of requests come up most often near campus?

The cell below puts the August records into a small database, as in Lab 5. The table is called `august`. `sql("""...""")` runs a query and returns the answer as a pandas table.

In [ ]:
# Supplied: put the August records in a database table called "august".
database = sqlite3.connect(":memory:")
csv_rows.to_sql("august", database, index=False)

def sql(query):
    return pd.read_sql_query(query, database)

sql("SELECT * FROM august LIMIT 3")

**Your turn:**

1. Write a query for the 8 most common request types. It should return two columns: `case_request` and `requests` (the number of cases of that type). Put the largest count first. If two counts are the same, sort those rows by `case_request` from A to Z. Save the result as `request_summary`.
2. Add a `share` column to `request_summary` in pandas: `requests` divided by the number of rows in the whole August table.
3. Now do the same for open cases only (`case_status` is `'Open'`). Copy your query, add one line, and keep the top 5. Save the result as `open_summary`. Add a `share` column here too, but divide by the number of open cases. Get that number with a separate query.

Then run the check cell. It counts the same things in pandas and compares.

In [ ]:
# Your turn: request_summary (the query, then the share column)
request_summary = ...


# Your turn: open_summary (the query, the number of open cases, then the share column)
open_summary = ...

In [ ]:
# Check: pandas should give the same counts as your SQL.
pandas_counts = csv_rows["case_request"].value_counts()
pandas_open_counts = csv_rows.loc[csv_rows["case_status"] == "Open", "case_request"].value_counts()

assert list(request_summary.columns) == ["case_request", "requests", "share"]
assert request_summary["requests"].tolist() == pandas_counts.head(8).tolist()
assert open_summary["requests"].tolist() == pandas_open_counts.head(5).tolist()
print("SQL and pandas agree.")

### Your response


Answer in two or three sentences: in `open_summary`, what number is `share` divided by? Why might the share of open cases be different from the share of all cases? Give one reason.


> Write your response here, then delete this line.

## 4. Join two downloads

You have two downloads of the same request. The August table (from Task 3) was downloaded on August 23, 2026. The September table was downloaded on September 23, 2026. The September table has fewer cases. Join the two tables on `case_id` to find out which cases are missing and why.

A `LEFT JOIN` keeps every row of the first table. If a row has no match in the second table, the second table's columns are empty (`NULL`). To find the rows with no match, look for rows where the second table's `case_id` `IS NULL`.

In [ ]:
# Supplied: load the September download and add it to the database as "september".
refresh_rows = pd.read_csv(course_data_path("hw3_campus_edge_311_2025_refresh.csv"), dtype={"case_id": str})
refresh_rows.to_sql("september", database, index=False, if_exists="replace")
print(len(csv_rows), "cases in the August table")
print(len(refresh_rows), "cases in the September table")

In [ ]:
# Worked example: cases that are in September but not in August.
# Start FROM september, so every September case is kept.
# If August has no matching case, a.case_id is NULL.
new_cases = sql("""
    SELECT s.case_id, s.date_opened, s.case_status, s.last_action_date
    FROM september AS s
    LEFT JOIN august AS a
        ON s.case_id = a.case_id
    WHERE a.case_id IS NULL
""")
print(len(new_cases), "new case(s)")
new_cases

**Your turn:**

1. Do the opposite of the worked example. Make `disappeared`: the `case_id`, `case_status`, and `last_action_date` of every August case that is not in September. How many are there?
2. Find out why these cases disappeared. Write two queries: one for the latest `last_action_date` among the cases in `disappeared`, and one for the earliest `last_action_date` in the September table. Use `MAX(column)` and `MIN(column)`. The dates are saved as text, like `2025-09-18T00:00:00.000`. Text in year-month-day order sorts by date, so `MIN` and `MAX` work.

Then run the check cell. It does the same comparison in pandas.

In [ ]:
# Your turn 1: disappeared
disappeared = ...


# Your turn 2: the two dates


In [ ]:
# Check: the same comparison in pandas. You don't need to change this cell.
# how="outer" keeps every case from both tables. indicator=True adds a _merge column
# that says whether a case is in both tables, only the left (August), or only the right (September).
comparison = csv_rows.merge(refresh_rows, on="case_id", how="outer", indicator=True)
merge_counts = comparison["_merge"].value_counts()
print(merge_counts)

assert len(disappeared) == merge_counts["left_only"]
assert len(new_cases) == merge_counts["right_only"]
assert len(csv_rows) == merge_counts["both"] + len(disappeared)
assert len(refresh_rows) == merge_counts["both"] + len(new_cases)
print("Your SQL matches pandas, and every case is accounted for.")

### Your response


Answer in three or four sentences:

1. Why did these cases disappear from the September download? Use your two dates as evidence.
2. Your `request_summary` in Task 3 used the August table. If someone ran the same request today, would they get the same numbers? What should a report based on this kind of data write down so that others can check it?


> Write your response here, then delete this line.